In [3]:
import os
import sys
import re
from statistics import mean

In [4]:
def extract_map50_from_file(filepath):
    """Extract the mAP50 value for the 'all' row from a text file."""
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip().startswith("all"):
                parts = line.split()
                if len(parts) >= 6:
                    return float(parts[5])  # mAP50 is 6th column
    return None

In [8]:
def extract_map50_classes(filepath, class_list):
    selected_values = []

    with open(filepath, "r") as f:
        for line in f:
            parts = line.strip().split()

            # Skip any line that doesn't contain 6 numeric trailing columns
            if len(parts) < 7:
                continue

            # Extract numeric columns safely
            try:
                images = int(parts[-6])
                instances = int(parts[-5])
                P = float(parts[-4])
                R = float(parts[-3])
                mAP50 = float(parts[-2])
                mAP5095 = float(parts[-1])
            except ValueError:
                # Not a valid data row
                continue

            # Reconstruct class name (may contain spaces)
            class_name = " ".join(parts[:-6])

            # If this class is in the user’s requested subset, save its mAP50
            if class_name in class_list:
                selected_values.append(mAP50)

    # Compute average if we found any values
    if not selected_values:
        return None  # or raise an error if preferred

    return sum(selected_values) / len(selected_values)


In [9]:
splits = {
    "Split A1": [
        "baseballfield",
        "basketballcourt",
        "bridge",
        "chimney",
        "ship"
    ],
    "Split A2": [
        "airplane",
        "airport",
        "Expressway-toll-station",
        "harbor",
        "groundtrackfield"
    ],
    "Split A3": [
        "dam",
        "golffield",
        "storagetank",
        "tenniscourt",
        "vehicle"
    ],
    "Split A4": [
        "Expressway-Service-area",
        "overpass",
        "stadium",
        "trainstation",
        "windmill"
    ],
    "Split B": [
        "airplane",
        "baseballfield",
        "tenniscourt",
        "trainstation",
        "windmill"
    ],
    "Split C1": [
        "trainstation",
        "harbor",
        "golffield",
        "airport",
        "storagetank"
    ],
    "Split C2": [
        "harbor",
        "overpass",
        "chimney",
        "Expressway-toll-station",
        "bridge"
    ],
    "Split C3": [
        "groundtrackfield",
        "basketballcourt",
        "harbor",
        "overpass",
        "tenniscourt"
    ]
}

In [11]:
root_dir = 'run/eval/detection/dior/coop_prototypes/backbone_remoteclip-14_boxes'

pattern = re.compile(r"N(\d+)-(\d+)")
data = {}  # {N: {seed: value}}

for folder_name in os.listdir(root_dir):
    folder_path = os.path.join(root_dir, folder_name)
    if os.path.isdir(folder_path):
        match = pattern.match(folder_name)
        if match:
            N = int(match.group(1))
            seed = int(match.group(2))

            txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]
            if not txt_files:
                continue

            # assume first txt file is the metrics file
            txt_path = os.path.join(folder_path, txt_files[0])

            # compute mAP50 for each split
            split_vals = {}
            for split_name, class_list in splits.items():
                val = extract_map50_classes(txt_path, class_list)
                if val is not None:
                    split_vals[split_name] = val

            if split_vals:
                data.setdefault(N, {})[seed] = split_vals

# --- Print results: per N, per seed, per split, plus averages over seeds ---
split_names = list(splits.keys())

for N in sorted(data.keys()):
    print(f"\n=== N={N} ===")

    # ---------- compute column widths ----------
    # Start with header lengths
    col_widths = {"Seed": len("Seed")}
    for split_name in split_names:
        col_widths[split_name] = len(split_name)

    # Update widths based on values
    for seed in data[N].keys():
        col_widths["Seed"] = max(col_widths["Seed"], len(str(seed)))
        for split_name in split_names:
            val = data[N][seed].get(split_name)
            s = f"{val:.4f}" if val is not None else "----"
            col_widths[split_name] = max(col_widths[split_name], len(s))

    # ---------- helpers ----------
    def fmt(col_name, value):
        """Left-align strings in a fixed-width column."""
        width = col_widths[col_name] + 2  # add some padding
        return str(value).ljust(width)

    # ---------- header ----------
    header = fmt("Seed", "Seed") + "".join(
        fmt(split_name, split_name) for split_name in split_names
    )
    print(header)

    # ---------- per-seed rows ----------
    for seed in sorted(data[N].keys()):
        row = fmt("Seed", seed)
        for split_name in split_names:
            val = data[N][seed].get(split_name)
            cell = f"{val:.4f}" if val is not None else "----"
            row += fmt(split_name, cell)
        print(row)

    # ---------- averages row ----------
    avg_row = fmt("Seed", "avg")
    for split_name in split_names:
        vals = [
            data[N][seed].get(split_name)
            for seed in data[N]
            if data[N][seed].get(split_name) is not None
        ]
        cell = f"{mean(vals):.4f}" if vals else "----"
        avg_row += fmt(split_name, cell)
    print(avg_row)


=== N=1 ===
Seed  Split A1  Split A2  Split A3  Split A4  Split B  Split C1  Split C2  Split C3  
1     0.3462    0.1613    0.2486    0.0506    0.3161   0.1018    0.0361    0.2169    
2     0.3845    0.1598    0.2328    0.0504    0.3156   0.0777    0.0380    0.2456    
3     0.3450    0.1778    0.2593    0.0564    0.3322   0.0972    0.0270    0.2344    
4     0.4158    0.1727    0.2704    0.0581    0.3392   0.0992    0.0670    0.2548    
5     0.3635    0.1632    0.2574    0.0582    0.3311   0.0991    0.0335    0.2462    
avg   0.3710    0.1669    0.2537    0.0547    0.3269   0.0950    0.0403    0.2396    

=== N=3 ===
Seed  Split A1  Split A2  Split A3  Split A4  Split B  Split C1  Split C2  Split C3  
1     0.3807    0.1664    0.2499    0.0508    0.3284   0.0885    0.0447    0.2371    
2     0.3836    0.1765    0.2370    0.0520    0.3339   0.0852    0.0404    0.2489    
3     0.3503    0.1812    0.2614    0.0561    0.3372   0.1061    0.0213    0.2391    
4     0.3932    0.1735    0.

In [5]:
root_dir = 'run/eval/detection/dior/coop_prototypes/backbone_remoteclip-14_boxes'

pattern = re.compile(r"N(\d+)-(\d+)")
data = {}  # {N: {seed: value}}

for folder_name in os.listdir(root_dir):
    folder_path = os.path.join(root_dir, folder_name)
    if os.path.isdir(folder_path):
        match = pattern.match(folder_name)
        if match:
            N = int(match.group(1))
            seed = int(match.group(2))
            txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]
            if not txt_files:
                continue
            txt_path = os.path.join(folder_path, txt_files[0])
            #val = extract_map50_from_file(txt_path)
            
            if val is not None:
                data.setdefault(N, {})[seed] = val

# Print in sorted order
for N in sorted(data.keys()):
    print(f"\n=== N={N} ===")
    for seed in sorted(data[N].keys()):
        val = data[N][seed]
        #print(f"Seed {seed}: mAP50 = {val:.4f}")
        print(f'{val:.4f}')
    avg_val = mean(data[N].values())
    print(f"Average for N={N}: {avg_val:.5f}")


=== N=1 ===
0.2017
0.2069
0.2096
0.2293
0.2106
Average for N=1: 0.21162

=== N=3 ===
0.2120
0.2123
0.2123
0.2236
0.2313
Average for N=3: 0.21830

=== N=5 ===
0.2182
0.2183
0.2058
0.2385
0.2170
Average for N=5: 0.21956

=== N=10 ===
0.2196
0.2128
0.2295
0.2431
0.2130
Average for N=10: 0.22360

=== N=30 ===
0.2171
0.2133
0.2152
0.2170
0.2054
Average for N=30: 0.21360
